
# Anchors: A Local Explanation Method

This interactive notebook illustrates the concept of **Anchors**, a method for generating local, rule-based explanations for black-box classifiers via random perturbations.

Specifically, we will:
1. Define a simple text classifier.
2. Introduce **anchors** as subsets of words in the input.
3. Estimate **precision** of an anchor via Monte Carlo sampling.
4. Explore **coverage** over a small synthetic corpus.
5. Visualize precision vs anchor size and demonstrate how to select anchors with high precision and minimal size.


Anchor sizes (|A|) will range from 1 to 10 on the x-axis.


In [ ]:
import itertools
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
while not (ROOT / "xai_book").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from xai_book.plotting import (
    BLUE as bookBlue,
    CHAPTER_GRAY as chaptergray,
    ORANGE as bookOrange,
    apply_notebook_style,
)

apply_notebook_style(figsize=(8.0, 5.0))

random.seed(0)
np.random.seed(0)


In [2]:

# Define a simple binary text classifier
def classify(doc_tokens):
    """Returns 1 if the word 'good' or 'great' appears in the tokens, else 0."""
    return int(any(word in ['good', 'great'] for word in doc_tokens))

# Example document
doc = "The service is not bad, prices are fine, and the food is great".split()
print("Document tokens:", doc)
print("Classifier prediction:", classify(doc))


Document tokens: ['The', 'service', 'is', 'not', 'bad,', 'prices', 'are', 'fine,', 'and', 'the', 'food', 'is', 'great']
Classifier prediction: 1


In [3]:

def perturb(tokens, anchor, keep_prob=0.5):
    """Generate a perturbed doc keeping all anchor words and randomly dropping others."""
    return [w for w in tokens if w in anchor or random.random() < keep_prob]

def estimate_precision(tokens, anchor, n_samples=100):
    correct = sum(classify(perturb(tokens, anchor)) == classify(tokens) for _ in range(n_samples))
    return correct / n_samples

# Quick test
anchor_example = ('food', 'great')
print(f"Estimated precision for anchor {anchor_example}: {estimate_precision(doc, anchor_example):.2f}")


Estimated precision for anchor ('food', 'great'): 1.00


In [ ]:
# Generate anchors of sizes 1 to 10 (sample up to 30 anchors per size)
anchors = []
for size in range(1, 11):
    combos = list(itertools.combinations(doc, size))
    random.shuffle(combos)
    anchors.extend(combos[:30])

precisions = [estimate_precision(doc, anchor) for anchor in anchors]
percent_prec = [precision * 100 for precision in precisions]
sizes = [len(anchor) for anchor in anchors]

threshold = 0.95
plt.figure(figsize=(8, 5))
plt.scatter(sizes, percent_prec, color=bookBlue, alpha=0.7, label='Anchors')
plt.axhline(y=threshold * 100, color=bookOrange, linestyle='--', label='Threshold (95%)')

best_anchors = [
    (sizes[i], percent_prec[i])
    for i in range(len(anchors))
    if precisions[i] >= threshold
]
min_size = min(size for size, _ in best_anchors)
best_points = [(size, precision) for size, precision in best_anchors if size == min_size]
xs, ys = zip(*best_points)
plt.scatter(xs, ys, color=chaptergray, s=100, marker='D', label='Selected Anchors')

plt.xlabel('Anchor Size |A|')
plt.ylabel('Estimated Precision (%)')
plt.title('Precision vs Anchor Size')
plt.xticks(range(1, 11))
plt.yticks(range(0, 101, 10))
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Simulate a small corpus of 200 documents from the toy vocabulary
vocab = list(set(doc))
corpus = [random.choices(vocab, k=random.randint(5, 10)) for _ in range(200)]

def coverage(anchor):
    return sum(all(word in document for word in anchor) for document in corpus) / len(corpus)

covs = [coverage(anchor) for anchor in anchors]
percent_cov = [coverage_value * 100 for coverage_value in covs]

plt.figure(figsize=(8, 5))
plt.scatter(sizes, percent_cov, color=bookBlue, alpha=0.7)
plt.xlabel('Anchor Size |A|')
plt.ylabel('Coverage (%)')
plt.title('Coverage vs Anchor Size')
plt.xticks(range(1, 11))
plt.yticks(range(0, 101, 10))
plt.grid(True)
plt.show()


In [6]:

# Print anchors with precision >= threshold
threshold = 0.95
# Collect anchors and precisions from previous computations
valid_anchors = [(anchors[i], sizes[i], percent_prec[i]) 
                 for i in range(len(anchors)) if precisions[i] >= threshold]
print(f"Anchors above the {threshold*100:.0f}% threshold:")
for anchor, size, prec in valid_anchors:
    print(f"Anchor: {anchor}, Size: {size}, Precision: {prec:.0f}%")


Anchors above the 95% threshold:
Anchor: ('great',), Size: 1, Precision: 100%
Anchor: ('fine,', 'great'), Size: 2, Precision: 100%
Anchor: ('food', 'great'), Size: 2, Precision: 100%
Anchor: ('not', 'great'), Size: 2, Precision: 100%
Anchor: ('are', 'great'), Size: 2, Precision: 100%
Anchor: ('are', 'food', 'great'), Size: 3, Precision: 100%
Anchor: ('The', 'is', 'great'), Size: 3, Precision: 100%
Anchor: ('are', 'and', 'great'), Size: 3, Precision: 100%
Anchor: ('not', 'are', 'great'), Size: 3, Precision: 100%
Anchor: ('bad,', 'prices', 'great'), Size: 3, Precision: 100%
Anchor: ('fine,', 'is', 'great'), Size: 3, Precision: 100%
Anchor: ('fine,', 'the', 'great'), Size: 3, Precision: 100%
Anchor: ('service', 'fine,', 'is', 'great'), Size: 4, Precision: 100%
Anchor: ('service', 'is', 'the', 'great'), Size: 4, Precision: 100%
Anchor: ('not', 'bad,', 'fine,', 'great'), Size: 4, Precision: 100%
Anchor: ('bad,', 'prices', 'the', 'great'), Size: 4, Precision: 100%
Anchor: ('not', 'prices', '

## Conclusion

- Anchor sizes range from 1 to 10 on the x-axis (whole numbers).
- Precision estimates use Monte Carlo sampling (100 runs) and are visualized in the shared muted book blue.
- The 95% precision threshold is drawn in the shared muted book orange and selected anchors are highlighted in chapter gray.
- Coverage across a synthetic corpus is plotted in the same blue to keep the chapter style consistent.

This setup clearly shows how anchor size trades off precision and coverage, with intuitive, rule-based explanations for the classifier.
